# ComfortPy LLM-Native Integration Tutorial

This notebook demonstrates how to use ComfortPy as a **deterministic tool layer** for LLM agents — semantic serialization, function-calling schemas, and prompt templates.

| Section | Topic | Extra Required |
|---------|-------|---------------|
| 1 | Semantic interpreters (markdown + summary dict) | — |
| 2 | Markdown summary from raw sensor DataFrame | — |
| 3 | Prompt templates for edge deployment | — |
| 4 | Pydantic tool schemas (function calling) | `[agent]` |
| 5 | OpenAI tool definitions | `[agent]` |
| 6 | LangChain adapter | `[agent]` + langchain |

**Install**: `pip install comfortpy[agent]` for sections 4–6. Sections 1–3 work with core only.

In [1]:
import numpy as np
import pandas as pd
import json
import traceback

from comfortpy import (
    evaluate_thermal,
    evaluate_visual,
    evaluate_acoustic,
    evaluate_iaq,
    calculate_global_ieq,
    calculate_compliance,
    ieq_to_markdown,
    ieq_to_summary_dict,
    generate_markdown_summary,
    EDGE_SYSTEM_PROMPT,
    DIAGNOSTIC_PROMPT_TEMPLATE,
    format_prompt,
)

# Simulate 24 hours of sensor data
rng = np.random.default_rng(42)
n = 24
df = pd.DataFrame({
    "temperature": 23 + rng.normal(0, 1.5, n),
    "radiant_temp": 23 + rng.normal(0, 1.5, n),
    "rh": 50 + rng.normal(0, 5, n),
    "air_speed": 0.1 + rng.normal(0, 0.02, n),
    "lux": 450 + rng.normal(0, 80, n),
    "laeq": 45 + rng.normal(0, 6, n),
    "co2": 700 + rng.normal(0, 120, n),
})

# Evaluate all domains
thermal = evaluate_thermal(
    tdb=df["temperature"].values, tr=df["radiant_temp"].values,
    vr=df["air_speed"].values, rh=df["rh"].values,
    met=1.2, clo=0.5, category="B",
)
visual = evaluate_visual(illuminance=df["lux"].values, task_type="office_writing")
acoustic = evaluate_acoustic(laeq=df["laeq"].values, nc_level="NC-35")
iaq = evaluate_iaq(co2=df["co2"].values, threshold_level="good")

ieq_result = calculate_global_ieq(
    thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq,
)
report = calculate_compliance(ieq_result, threshold=70.0)

print(f"IEQ Index avg: {np.mean(ieq_result.index):.1f}")
print(f"Compliance: {report.compliance_rate_pct:.1f}%")

IEQ Index avg: 75.5
Compliance: 79.2%


## 1. Semantic Interpreters

`ieq_to_markdown()` converts a `GlobalIEQResult` into token-efficient structured markdown (~200 tokens vs 14,400 for raw 24h data). `ieq_to_summary_dict()` returns a flat dict for structured JSON injection.

In [2]:
# Markdown output (for LLM context injection)
md = ieq_to_markdown(ieq_result, compliance_report=report, zone_id="Room-402")
print(md)
print(f"\n--- Token estimate: ~{len(md) // 4} tokens ({len(md)} chars) ---")

### Building System Report: Zone Room-402
- **Current Operational State**: NON-COMPLIANT
- **Global IEQ Index Score**: 75.5/100 (min: 61.7, max: 88.9)
- **Contract Compliance Rate**: 79.2%
- **Timestamps Evaluated**: 24

#### Domain Breakdown:
  - [OK] THERMAL: 85.6/100
  - [OK] VISUAL: 85.1/100
  - [WARNING] ACOUSTIC: 34.1/100
  - [OK] IAQ: 76.8/100

#### Diagnostic Insight:
The primary limiting factor is the **ACOUSTIC** domain (score: 34.1/100). Noise levels exceeded NC threshold. Action: inspect noise sources or add acoustic damping.

--- Token estimate: ~135 tokens (543 chars) ---


In [3]:
# Summary dict (for structured JSON context)
summary = ieq_to_summary_dict(ieq_result, compliance_report=report)
print("Summary dict (for JSON injection):")
print(json.dumps(summary, indent=2, default=str))

Summary dict (for JSON injection):
{
  "ieq_index_avg": 75.5,
  "ieq_index_min": 61.7,
  "ieq_index_max": 88.9,
  "ieq_index_std": 6.4,
  "n_timestamps": 24,
  "domains": [
    "thermal",
    "visual",
    "acoustic",
    "iaq"
  ],
  "domain_scores_avg": {
    "thermal": 85.6,
    "visual": 85.1,
    "acoustic": 34.1,
    "iaq": 76.8
  },
  "worst_domain": "acoustic",
  "weights_used": {
    "thermal": 0.4,
    "visual": 0.2,
    "acoustic": 0.15,
    "iaq": 0.25
  },
  "compliance_rate_pct": 79.2,
  "threshold": 70.0,
  "compliant_hours": 19.0,
  "total_hours": 24.0
}


## 2. Markdown Summary from Raw DataFrame

`generate_markdown_summary()` runs the full IEQ pipeline on a sensor DataFrame and produces a dense report — the foundation for RAG ingestion or historical analysis.

In [4]:
# Generate full markdown report from raw sensor DataFrame
report_md = generate_markdown_summary(df, threshold=70.0, zone_id="Room-402")
print(report_md)

## IEQ Report: Zone Room-402
* **Global IEQ Average:** 68.9/100
* **Compliance Rate:** 41.7% (Threshold > 70)
* **Timestamps:** 24

### Domain Summary
* **VISUAL:** avg=85.1/100, compliance=70.8%
* **ACOUSTIC:** avg=34.1/100, compliance=4.2%
* **IAQ:** avg=76.8/100, compliance=50.0%

### Critical Failures
* **14 timestamps** below threshold (IEQ < 70).
* Primary cause: **ACOUSTIC** domain (avg score during failures: 14.5/100).
* Noise levels exceeded NC threshold. Action: inspect noise sources or add acoustic damping.



## 3. Prompt Templates for Edge Deployment

Pre-built system prompts with physics-guard rules for deploying ComfortPy alongside local LLMs (Llama, Mistral) on edge gateways.

In [5]:
# Edge deployment system prompt with IEQ context injected
prompt = format_prompt(
    EDGE_SYSTEM_PROMPT,
    context=md,
    query="Occupants in Room-402 complain it's warm and stuffy. Diagnose and recommend actions.",
)
print(prompt)

You are an expert Edge Building Diagnostics Engine powered by ComfortPy.
You receive structured multi-domain physical summaries from localized sensors.

CRITICAL RULES:
1. Never invent physical metrics. Rely strictly on the passed ComfortPy tool outputs.
2. If PMV is high (positive) and CO₂ is low, do not recommend increasing outdoor air if ambient air temperature is above 30°C, as it will exacerbate the thermal discomfort.
3. Output your reasoning first, followed by an actionable BACnet/Modbus control command payload in clean JSON format.

[CONTEXT]
### Building System Report: Zone Room-402
- **Current Operational State**: NON-COMPLIANT
- **Global IEQ Index Score**: 75.5/100 (min: 61.7, max: 88.9)
- **Contract Compliance Rate**: 79.2%
- **Timestamps Evaluated**: 24

#### Domain Breakdown:
  - [OK] THERMAL: 85.6/100
  - [OK] VISUAL: 85.1/100
  - [WARNING] ACOUSTIC: 34.1/100
  - [OK] IAQ: 76.8/100

#### Diagnostic Insight:
The primary limiting factor is the **ACOUSTIC** domain (score: 3

In [6]:
# Diagnostic prompt template
diag_prompt = format_prompt(
    DIAGNOSTIC_PROMPT_TEMPLATE,
    ieq_report=md,
    complaint="Occupants feel drowsy and warm during afternoon hours.",
)
print(diag_prompt)

You are a building comfort diagnostic assistant. Based on the following IEQ report, diagnose the issue and recommend remediation actions.

IEQ Report:
### Building System Report: Zone Room-402
- **Current Operational State**: NON-COMPLIANT
- **Global IEQ Index Score**: 75.5/100 (min: 61.7, max: 88.9)
- **Contract Compliance Rate**: 79.2%
- **Timestamps Evaluated**: 24

#### Domain Breakdown:
  - [OK] THERMAL: 85.6/100
  - [OK] VISUAL: 85.1/100
  - [WARNING] ACOUSTIC: 34.1/100
  - [OK] IAQ: 76.8/100

#### Diagnostic Insight:
The primary limiting factor is the **ACOUSTIC** domain (score: 34.1/100). Noise levels exceeded NC threshold. Action: inspect noise sources or add acoustic damping.

Occupant Complaint:
Occupants feel drowsy and warm during afternoon hours.

Provide:
1. Root cause analysis (which domain is failing and why)
2. Immediate remediation steps
3. Long-term recommendations


## 4. Pydantic Tool Schemas (Function Calling)

`evaluate_thermal_tool()` and `evaluate_ieq_tool()` wrap ComfortPy's core calculations into LLM-callable functions with typed inputs (list[float]) and structured dict outputs. Requires `pip install comfortpy[agent]`.

In [7]:
def try_run(label, fn):
    """Run a function, catching ImportError for missing extras."""
    try:
        result = fn()
        print(f"  [OK] {label}")
        return result
    except ImportError as e:
        print(f"  [SKIP] {label}")
        print(f"        {e}")
        return None
    except Exception:
        print(f"  [ERROR] {label}")
        traceback.print_exc()
        return None

def demo_thermal_tool():
    from comfortpy.llm.tools import evaluate_thermal_tool

    result = evaluate_thermal_tool(
        air_temperature=[23.0, 24.0, 25.0, 26.0],
        relative_humidity=[50.0, 55.0, 60.0, 58.0],
    )
    print("evaluate_thermal_tool result:")
    print(json.dumps(result, indent=2))
    return result

thermal_tool_result = try_run("evaluate_thermal_tool", demo_thermal_tool)

evaluate_thermal_tool result:
{
  "mean_pmv": -0.02,
  "mean_ppd": 7.7,
  "max_ppd": 10.4,
  "discomfort_risk": "acceptable"
}
  [OK] evaluate_thermal_tool


In [8]:
def demo_ieq_tool():
    from comfortpy.llm.tools import evaluate_ieq_tool

    result = evaluate_ieq_tool(
        air_temperature=[23.0, 24.0, 25.0, 26.0],
        relative_humidity=[50.0, 55.0, 60.0, 58.0],
        illuminance=[500.0, 450.0, 480.0, 520.0],
        noise_laeq=[40.0, 42.0, 38.0, 45.0],
        co2=[800.0, 700.0, 750.0, 900.0],
    )
    print("evaluate_ieq_tool result:")
    # Print without markdown for brevity
    result_copy = {k: v for k, v in result.items() if k != "markdown"}
    print(json.dumps(result_copy, indent=2, default=str))
    print(f"\n--- Markdown ({len(result['markdown'])} chars) ---")
    print(result["markdown"])
    return result

ieq_tool_result = try_run("evaluate_ieq_tool", demo_ieq_tool)

evaluate_ieq_tool result:
{
  "ieq_index_avg": 79.9,
  "ieq_index_min": 74.0,
  "ieq_index_max": 84.8,
  "ieq_index_std": 3.9,
  "n_timestamps": 4,
  "domains": [
    "thermal",
    "visual",
    "acoustic",
    "iaq"
  ],
  "domain_scores_avg": {
    "thermal": 90.5,
    "visual": 96.3,
    "acoustic": 48.8,
    "iaq": 68.3
  },
  "worst_domain": "acoustic",
  "weights_used": {
    "thermal": 0.4,
    "visual": 0.2,
    "acoustic": 0.15,
    "iaq": 0.25
  }
}

--- Markdown (504 chars) ---
### Building System Report: Building
- **Current Operational State**: NON-COMPLIANT
- **Global IEQ Index Score**: 79.9/100 (min: 74.0, max: 84.8)
- **Timestamps Evaluated**: 4

#### Domain Breakdown:
  - [OK] THERMAL: 90.5/100
  - [OK] VISUAL: 96.3/100
  - [WARNING] ACOUSTIC: 48.8/100
  - [WARNING] IAQ: 68.3/100

#### Diagnostic Insight:
The primary limiting factor is the **ACOUSTIC** domain (score: 48.8/100). Noise levels exceeded NC threshold. Action: inspect noise sources or add acoustic damping.


## 5. OpenAI Tool Definitions

`to_openai_tools()` generates JSON schema dicts compatible with OpenAI's function calling API.

In [9]:
def demo_openai_tools():
    from comfortpy.llm.tools import to_openai_tools

    tools = to_openai_tools()
    print(f"OpenAI tool definitions ({len(tools)} tools):\n")
    for t in tools:
        print(f"  {t['function']['name']}: {t['function']['description'][:80]}...")
    print(f"\nFull schema for evaluate_thermal:")
    print(json.dumps(tools[0], indent=2))
    return tools

openai_tools_result = try_run("to_openai_tools", demo_openai_tools)

OpenAI tool definitions (2 tools):

  evaluate_thermal: Calculate thermal comfort metrics (PMV, PPD) from air temperature, humidity, rad...
  evaluate_ieq: Evaluate multi-domain Indoor Environmental Quality (thermal, visual, acoustic, I...

Full schema for evaluate_thermal:
{
  "type": "function",
  "function": {
    "name": "evaluate_thermal",
    "description": "Calculate thermal comfort metrics (PMV, PPD) from air temperature, humidity, radiant temperature, and air velocity arrays.",
    "parameters": {
      "type": "object",
      "properties": {
        "air_temperature": {
          "description": "Dry-bulb air temperatures in Celsius from space sensors.",
          "items": {
            "type": "number"
          },
          "title": "Air Temperature",
          "type": "array"
        },
        "relative_humidity": {
          "description": "Relative humidity percentages (0-100).",
          "items": {
            "type": "number"
          },
          "title": "Relative 

## 6. LangChain Adapter

`to_langchain_tools()` returns `StructuredTool` objects ready for LangChain agents. Requires `pip install langchain`.

In [10]:
def demo_langchain():
    from comfortpy.llm.tools import to_langchain_tools

    tools = to_langchain_tools()
    print(f"LangChain tools ({len(tools)}):")
    for t in tools:
        print(f"  {t.name}: {t.description[:80]}...")
    return tools

langchain_result = try_run("to_langchain_tools", demo_langchain)

  [SKIP] to_langchain_tools
        LangChain is required for to_langchain_tools(). Install with: pip install langchain


## Summary

### No extra dependencies (core only)
- `ieq_to_markdown()` — token-efficient markdown from `GlobalIEQResult` (~200 tokens)
- `ieq_to_summary_dict()` — flat dict for structured JSON context
- `generate_markdown_summary()` — full IEQ pipeline on DataFrame → markdown report
- `EDGE_SYSTEM_PROMPT` / `DIAGNOSTIC_PROMPT_TEMPLATE` — guarded prompt templates
- `format_prompt()` — simple template formatter

### Requires `pip install comfortpy[agent]`
- `evaluate_thermal_tool()` — LLM-callable thermal comfort evaluation
- `evaluate_ieq_tool()` — LLM-callable multi-domain IEQ evaluation
- `to_openai_tools()` — JSON schemas for OpenAI function calling
- `to_langchain_tools()` — `StructuredTool` objects for LangChain agents

### Architecture
```
src/comfortpy/llm/
├── interpreters.py   # No extra deps
├── prompts.py        # No extra deps
└── tools.py          # Requires pydantic (optional [agent] extra)
```

Core physics modules in `domains/` remain completely isolated from LLM logic.

# ComfortPy LLM-Native Integration Tutorial

This notebook demonstrates how to use ComfortPy as a **deterministic tool layer** for LLM agents — semantic serialization, function-calling schemas, and prompt templates.

| Section | Topic | Extra Required |
|---------|-------|---------------|
| 1 | Semantic interpreters (markdown + summary dict) | — |
| 2 | Markdown summary from raw sensor DataFrame | — |
| 3 | Prompt templates for edge deployment | — |
| 4 | Pydantic tool schemas (function calling) | `[agent]` |
| 5 | OpenAI tool definitions | `[agent]` |
| 6 | LangChain adapter | `[agent]` + langchain |

**Install**: `pip install comfortpy[agent]` for sections 4–6. Sections 1–3 work with core only.